In [13]:
import os
import glob
from datetime import datetime
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Border, Alignment

# --- CONFIGURATION ---
INPUT_FOLDER = "./data"  # Change to your folder path
OUTPUT_FILE = "Master_Unpivoted_Attendance_Report.xlsx"

print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Environment initialized.")
print(f"Targeting folder: {INPUT_FOLDER}")

[2026-05-20 23:00:25] Environment initialized.
Targeting folder: ./data


In [14]:
from pathlib import Path

def get_all_excel_paths_recursive(main_folder_path):
    """
    Recursively searches through the main folder and all subfolders 
    for valid Excel files (.xlsx, .xlsm).
    
    Parameters:
    main_folder_path (str): The root directory to start the search from.
    
    Returns:
    list: A list of string paths to matching Excel files.
    """
    # Convert input string to a Path object
    root_dir = Path(main_folder_path)
    
    if not root_dir.exists():
        print(f"[ERROR] The directory path '{main_folder_path}' does not exist.")
        return []
        
    excel_extensions = {'.xlsx', '.xlsm'}
    discovered_paths = []
    
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] starting recursive scan in: {root_dir.resolve()}")
    
    # rglob("*") recursively yields all files and directories under the root path
    for path in root_dir.rglob("*"):
        # Ensure it's a file, matches our target extensions, and isn't an Excel temporary/lock file
        if path.is_file() and path.suffix.lower() in excel_extensions:
            if not path.name.startswith("~$"):
                # Append the absolute path as a string to match your existing functions
                discovered_paths.append(str(path.resolve()))
                
    print(f"Scan complete. Discovered {len(discovered_paths)} potential Excel files across all subfolders.\n")
    return discovered_paths

In [15]:
# Modified Cell 2 function signature
def analyze_and_filter_files(file_path_list, target_tab='all'):
    valid_files = []
    
    # Directly loop through the list passed into the function
    for file_path in file_path_list:
        # Ignore our own output file if it happens to be in the scan path
        if os.path.basename(file_path) == OUTPUT_FILE:
            continue
        try:
            wb = openpyxl.load_workbook(file_path, data_only=True)
            
            # 1. Case-insensitive "all" tab check (matches 'all', 'All', 'ALL', etc.)
            target_sheet_name = None
            for sheet_name in wb.sheetnames:
                if sheet_name.lower().strip() == target_tab:
                    target_sheet_name = sheet_name
                    break
            
            if not target_sheet_name:
                continue  # Skip file if no variation of "all" is found
                
            ws = wb[target_sheet_name]
            file_name = os.path.basename(file_path)
            
            valid_files.append(file_path)
            wb.close()
            
        except Exception as e:
            print(f"Error evaluating file {os.path.basename(file_path)}: {str(e)}")
            
    return valid_files

In [18]:
aux=get_all_excel_paths_recursive(INPUT_FOLDER)
len(aux)

[2026-05-20 23:18:04] starting recursive scan in: /home/ubuntu/Desktop/GCL/gcl_personnel/data
Scan complete. Discovered 18 potential Excel files across all subfolders.



18

In [19]:
import re
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

def find_all_sheet(workbook):
    """Finds the 'All' tab regardless of case variations (ALL, All, all)."""
    for sheet_name in workbook.sheetnames:
        if sheet_name.lower().strip() == "all":
            return workbook[sheet_name]
    return None

In [20]:
def extract_cell_value_and_color(cell):
    """
    Extracts text value and checks if senior staff highlighted the cell.
    Converts visual shapes or specific markers to standard text strings.
    """
    val = cell.value
    if val is None:
        return None, False
        
    # Standardize string formatting
    val_str = str(val).strip().lower()
    
    # Map symbols to clear database strings (Modify these characters if needed)
    if val_str in ["○", "circle", "o"]:
        availability = "online & in person"
    elif "online" in val_str:
        availability = "online"
    elif "in person" in val_str or "in-person" in val_str:
        availability = "in person"
    else:
        availability = str(val).strip() # Fallback for 'X' or other text markers

    # Check for senior staff background color fills
    is_approved = False
    if cell.fill and cell.fill.start_color:
        color_hex = cell.fill.start_color.rgb
        # Filter out default uncolored grids (openpyxl often returns '00000000' or 0)
        if color_hex and color_hex != "00000000" and color_hex != 0:
            is_approved = True
            
    return availability, is_approved

In [21]:
def parse_time_block(time_str):
    """Splits a string range like '10:00-10:30' into separate clean time parts."""
    if not time_str or "-" not in str(time_str):
        return None, None
    try:
        parts = str(time_str).split("-")
        start = parts[0].strip() + ":00"
        end = parts[1].strip() + ":00"
        return start, end
    except Exception:
        return None, None

In [39]:
# def find_grid_boundaries(sheet):
#     """
#     Scans the worksheet to locate the top-left anchor point dynamically.
#     Returns the student header row index, date column index, and time column index.
#     """
#     # Regex pattern to match time blocks like 9:00-9:30 or 10:30-11:00
#     time_pattern = re.compile(r"^\d{1,2}:\d{2}-\d{1,2}:\d{2}$")
    
#     for row_idx in range(1, 15): # Scan first 15 rows for the anchor
#         for col_idx in range(1, 10): # Scan first 10 columns
#             cell_val = str(sheet.cell(row=row_idx, column=col_idx).value or "").strip()
#             if time_pattern.match(cell_val):
#                 # The time row confirms our structure. 
#                 # Student names sit in the row directly above this one.
#                 student_header_row = row_idx - 1
#                 date_col_idx = col_idx - 1
#                 time_col_idx = col_idx
#                 return student_header_row, date_col_idx, time_col_idx
                
#     raise ValueError("Could not find a valid time block (e.g. 9:00-9:30) to anchor the data grid.")

def find_grid_boundaries(sheet):
    """
    Locates the time column using a regex pattern. Then, scans to the left 
    to find the true Date column, ignoring any weekday text columns in between.
    """
    time_pattern = re.compile(r"^\d{1,2}:\d{2}-\d{1,2}:\d{2}$")
    
    # 1. Find the time column first
    for row_idx in range(1, 20):  # Scans up to row 20
        for col_idx in range(1, 15):  # Scans up to column 15
            cell_val = str(sheet.cell(row=row_idx, column=col_idx).value or "").strip()
            
            if time_pattern.match(cell_val):
                time_col_idx = col_idx
                student_header_row = row_idx - 1
                
                # 2. Scan BACKWARDS from the time column to locate the Date column
                date_col_idx = None
                for look_back in range(time_col_idx - 1, 0, -1):
                    test_val = sheet.cell(row=row_idx, column=look_back).value
                    
                    # If the cell is empty due to a vertical merge, we check rows above it
                    # within reasonable bounds to see if it belongs to a date column block.
                    check_row = row_idx
                    while test_val is None and check_row > student_header_row:
                        check_row -= 1
                        test_val = sheet.cell(row=check_row, column=look_back).value
                    
                    if test_val is not None:
                        test_str = str(test_val).strip().lower()
                        # Skip common day names and abbreviations
                        if test_str in ['monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
                                        'mon', 'tue', 'wed', 'thu', 'fri', 'sat', 'sun']:
                            continue
                        
                        # If it's not a day string, this is our true Date column anchor!
                        date_col_idx = look_back
                        break
                
                # Fallback safeguard: if no date column found, assume it's the immediate left
                if date_col_idx is None:
                    date_col_idx = time_col_idx - 1
                    
                return student_header_row, date_col_idx, time_col_idx
                
    raise ValueError("Could not find a valid time block (e.g. 9:00-9:30) to anchor the data grid.")

In [40]:
def process_all_workbooks(folder_path):
    """Main pipeline loop over all 19 files."""
    # Convert input string to a Path object
    master_records = []
    root_dir = Path(folder_path)
    
    if not root_dir.exists():
        print(f"[ERROR] The directory path '{folder_path}' does not exist.")
        return []
        
    excel_extensions = {'.xlsx', '.xlsm'}
    discovered_paths = []
    
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] starting recursive scan in: {root_dir.resolve()}")
    
    # rglob("*") recursively yields all files and directories under the root path
    for path in root_dir.rglob("*"):
        # Ensure it's a file, matches our target extensions, and isn't an Excel temporary/lock file
        if path.is_file() and path.suffix.lower() in excel_extensions:
            if not path.name.startswith("~$"):
                # Append the absolute path as a string to match your existing functions
                discovered_paths.append(str(path.resolve()))
                
    print(f"Scan complete. Discovered {len(discovered_paths)} potential Excel files across all subfolders.\n")
    
    for file_path in discovered_paths:
        print(f"Processing: {file_path}...")
        
        # Load workbook with data_only=True to get raw evaluation values instead of raw formulas
        wb = load_workbook(file_path, data_only=True)
        sheet = find_all_sheet(wb)
        
        if sheet is None:
            print(f" -> Skipping: No 'All' sheet found in {file_path}")
            continue
            
        try:
            student_row, date_col, time_col = find_grid_boundaries(sheet)
        except ValueError as e:
            print(f" -> Skipping {file_path}: {e}")
            continue

        # Extract all student names across the header row starting after the time column
        students = {}
        max_col = sheet.max_column
        for c in range(time_col + 1, max_col + 1):
            name = sheet.cell(row=student_row, column=c).value
            if name:
                students[c] = str(name).strip()

        # Step through rows down the schedule grid
        max_row = sheet.max_row
        current_date = None # Track merged date across iterations
        
        for r in range(student_row + 1, max_row + 1):
            # Resolve merged or standalone cell value for Date
            date_cell = sheet.cell(row=r, column=date_col)
            # Handle openpyxl merged cell tracking lookup automatically
            if date_cell.coordinate in sheet.merged_cells:
                # Find the top-left cell of the merge range to extract value safely
                for merge_range in sheet.merged_cells.ranges:
                    if date_cell.coordinate in merge_range:
                        date_cell = sheet.cell(row=merge_range.min_row, column=merge_range.min_col)
                        break
            
            if date_cell.value is not None:
                # Format to database friendly YYYY-MM-DD
                try:
                    current_date = pd.to_datetime(date_cell.value).strftime('%Y-%m-%d')
                except Exception:
                    current_date = str(date_cell.value).strip()

            time_cell_val = str(sheet.cell(row=r, column=time_col).value or "").strip()
            start_t, end_t = parse_time_block(time_cell_val)
            
            # If there's no valid time layout on this row, skip it (structural noise)
            if not start_t:
                continue

            # Loop across the registered students columns for this row
            for col_idx, student_name in students.items():
                cell = sheet.cell(row=r, column=col_idx)
                availability, is_approved = extract_cell_value_and_color(cell)
                
                # Filter out blank rows entirely to minimize database bloat
                if availability is None:
                    continue
                    
                master_records.append({
                    "source_file": file_path,
                    "event_date": current_date,
                    "start_time": start_t,
                    "end_time": end_t,
                    "student_name": student_name,
                    "availability": availability,
                    "is_approved": is_approved
                })
                
    # Build a clean database ready DataFrame
    master_df = pd.DataFrame(master_records)
    return master_df



In [ ]:
# --- Execution Entry Point ---
if __name__ == "__main__":
    # Point this path to the directory containing your 19 Excel files
    INPUT_FOLDER = "./data" 
    
    final_data = process_all_workbooks(INPUT_FOLDER)

[2026-05-20 23:35:41] starting recursive scan in: /home/ubuntu/Desktop/GCL/gcl_personnel/data
Scan complete. Discovered 18 potential Excel files across all subfolders.

Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.08_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R8.3_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.10_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.12_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.09_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.07_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.11_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/Schedule/2025/R7.06_GCL_Schedule.xlsm...
Processing: /home/ubuntu/Desktop/GCL/gcl_personnel/data/

In [43]:
# # Export to unified CSV file
final_data.to_csv("master_student_schedule.csv", index=False)
print(f"\nSuccessfully compiled {len(final_data)} neat data rows into master_student_schedule.csv!")


Successfully compiled 12655 neat data rows into master_student_schedule.csv!


In [42]:
final_data

,source_file,event_date,start_time,end_time,student_name,availability,is_approved
0,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2025-08-01,11:30:00,12:00:00,Pan,◯,False
1,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2025-08-01,11:30:00,12:00:00,Aimi,◯,True
2,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2025-08-01,12:00:00,12:30:00,Pan,◯,True
3,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2025-08-01,12:00:00,12:30:00,Aimi,◯,True
4,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2025-08-01,12:30:00,13:00:00,Pan,◯,True
...,...,...,...,...,...,...,...
12650,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2026-04-28,16:30:00,17:00:00,Sirpy,◯,True
12651,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2026-04-28,17:00:00,17:30:00,Jedidah,◯,True
12652,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2026-04-28,17:00:00,17:30:00,Sirpy,◯,True
12653,/home/ubuntu/Desktop/GCL/gcl_personnel/data/Sc...,2026-04-28,17:30:00,18:00:00,Jedidah,◯,True
